# Month-by-month diagnostic stability check

Reproduces **Table 4** in `paper/draft_paper.md` (Section 5.7, "Generalization and Temporal Stability of the Diagnostic Layer").

Runs `agent_diagnose.diagnose(zone, month=m)` independently for all 3 flows (BN, Middling, Reject) and all 5 available months (July–November 2025) — 15 flow-months total, no parameters shared or carried over between runs. `agent_demo.ipynb` only shows the November snapshot; this notebook is the source of the multi-month numbers cited in the paper, saved to `month_stability.csv` so the table can be checked independently of any prose claim.

In [1]:
import pandas as pd
from agent_diagnose import diagnose, resolve_zone

MONTHS = ['2025-07', '2025-08', '2025-09', '2025-10', '2025-11']
FLOWS = ['bn', 'middling', 'reject']

In [2]:
rows = []
for flow in FLOWS:
    zid = resolve_zone(flow)
    for m in MONTHS:
        dx = diagnose(zid, month=m)
        th = dx['throughput']
        cap = dx['capacity']
        dec = dx['decision']
        att = dx['attribution']
        top_bucket, top_bucket_h = max(att['recoverable_truck_h'].items(), key=lambda kv: kv[1])
        gain_pct = round((dec['ceiling_active'] / th['loads_day'] - 1) * 100, 1) if th['loads_day'] else None
        binding = 'shovel' if dec['binding_now'] == 'shovel (loading)' else 'fleet / schedule'
        rows.append(dict(
            flow=flow, month=m,
            loads_day=th['loads_day'], trucks=th['trucks'], days=th['days'], best_day=th['best_day'],
            shovel_util_pct=round(cap['utilisation'] * 100, 1),
            top_bucket=top_bucket, top_bucket_h=round(top_bucket_h, 1),
            binding_constraint=binding,
            ceiling_active=dec['ceiling_active'], shovel_ceiling=dec['shovel_ceiling'],
            recoverable_gain_pct=gain_pct,
        ))

df = pd.DataFrame(rows)
df

,flow,month,loads_day,trucks,days,best_day,shovel_util_pct,top_bucket,top_bucket_h,binding_constraint,ceiling_active,shovel_ceiling,recoverable_gain_pct
0,bn,2025-07,48.5,13,31,68,43.0,return_road,972,fleet / schedule,80,114,64.9
1,bn,2025-08,78.0,24,31,104,52.0,return_road,1535,fleet / schedule,125,150,60.3
2,bn,2025-09,77.6,24,30,106,53.0,return_road,1488,fleet / schedule,125,145,61.1
3,bn,2025-10,88.3,26,31,118,59.0,return_road,1669,fleet / schedule,136,150,54.0
4,bn,2025-11,80.6,22,30,109,50.0,return_road,1567,fleet / schedule,124,160,53.8
5,middling,2025-07,150.7,10,31,341,32.0,idle_onshift,612,fleet / schedule,233,477,54.6
6,middling,2025-08,196.7,25,31,413,33.0,idle_onshift,2786,fleet / schedule,404,587,105.4
7,middling,2025-09,233.3,27,30,410,44.0,idle_onshift,3115,fleet / schedule,484,532,107.5
8,middling,2025-10,204.6,29,31,508,38.0,idle_onshift,4186,fleet / schedule,471,541,130.2
9,middling,2025-11,159.9,21,30,387,34.0,idle_onshift,1826,fleet / schedule,299,477,87.0


In [3]:
# Reject July 2025: confirm the low volume is real activity, not a data-coverage gap
# (cited in draft_paper.md §5.7 as "158 recorded cycles, 9 active trucks, coverage on 22 of the month's 31 days")
from agent_diagnose import perception

cyc, _ = perception(resolve_zone('reject'), month='2025-07')
print('n cycles:', len(cyc))
print('n trucks:', cyc['truck_id'].nunique() if 'truck_id' in cyc.columns else cyc.iloc[:, 0].nunique())
print('n active days:', cyc['depart_load'].dt.date.nunique() if 'depart_load' in cyc.columns else 'n/a')
print('date range:', cyc['depart_load'].min(), '→', cyc['depart_load'].max())

n cycles: 158
n trucks: 9
n active days: 22
date range: 2025-07-02 07:42:18 → 2025-07-31 19:14:03


In [4]:
df.to_csv('month_stability.csv', index=False)
print('saved month_stability.csv:', len(df), 'rows')

saved month_stability.csv: 15 rows


## Cross-checks against the paper's claims (Section 5.7)

- **Shovel never binds**: `binding_constraint` should read `fleet / schedule` in all 15 rows, and `shovel_util_pct` should never reach 85.
- **BN stability**: `top_bucket` should be `return_road` in all 5 BN rows; `recoverable_gain_pct` should fall in a narrow band, with the November row the *lowest* of the five.
- **Middling**: `top_bucket` should be `idle_onshift` in all 5 rows, but `recoverable_gain_pct` should vary widely — this is the instability flagged as a limitation, not an error.
- **Reject**: `top_bucket` should be `load_queue` for Aug–Nov and `idle_onshift` for Jul; `loads_day` and `shovel_util_pct` for Jul should be far below the other four months.